In [1]:
import joblib
from pathlib import Path
import pandas as pd
from sklearn.preprocessing import StandardScaler

# 1. Định nghĩa đường dẫn root & output
BASE_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_RAW_PATH = BASE_DIR / 'data' / 'raw' / 'gas_turbine_data.csv'#chỉnh cho từng máy của từng thành viên

OUTPUT_DIR = BASE_DIR / 'data' / 'processed'
MODELS_DIR = BASE_DIR / 'models'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# 2. Đọc dữ liệu và loại bỏ bản ghi trùng lặp (Gán chuẩn tên biến df)
df_raw = pd.read_csv(DATA_RAW_PATH)
df = df_raw.drop_duplicates().copy()

In [2]:
train_df = df[df['year'].isin([2011, 2012, 2013])].copy()
test_df = df[df['year'].isin([2014, 2015])].copy()

In [3]:
# 4. Capping Outlier cột CO (tính p99 CHỈ trên tập Train để tránh Data Leakage)
co_p99_threshold = train_df['CO'].quantile(0.99)
train_df['CO_clipped'] = train_df['CO'].clip(upper=co_p99_threshold)

In [4]:
test_df['CO_clipped'] = test_df['CO'].clip(upper=co_p99_threshold)

# 5. Scaler chỉ fit trên 9 đặc trưng đầu vào của Train
feature_cols = ['AT', 'AP', 'AH', 'AFDP', 'GTEP', 'TIT', 'TAT', 'TEY', 'CDP']
scaler = StandardScaler()

train_df[feature_cols] = scaler.fit_transform(train_df[feature_cols])
test_df[feature_cols] = scaler.transform(test_df[feature_cols])

In [5]:
# 6. Lưu file processed và lưu artifacts (Scaler + Capping Threshold)
train_df.to_csv(OUTPUT_DIR / 'train_clean.csv', index=False)
test_df.to_csv(OUTPUT_DIR / 'test_clean.csv', index=False)

# Lưu Scaler & Threshold ra thư mục models/ để tái lập suy luận sau này
joblib.dump(scaler, MODELS_DIR / 'scaler.joblib')
joblib.dump({'co_p99_threshold': co_p99_threshold}, MODELS_DIR / 'preprocessing_meta.joblib')

print(f"[THÀNH CÔNG] Đã lưu train_clean.csv ({len(train_df)} dòng) & test_clean.csv ({len(test_df)} dòng)")
print(f"[THÀNH CÔNG] Ngưỡng p99 CO tính từ Train: {co_p99_threshold:.4f} mg/m³")
print(f"[THÀNH CÔNG] Đã đóng gói scaler.joblib tại {MODELS_DIR}")

[THÀNH CÔNG] Đã lưu train_clean.csv (22191 dòng) & test_clean.csv (14535 dòng)
[THÀNH CÔNG] Ngưỡng p99 CO tính từ Train: 10.7166 mg/m³
[THÀNH CÔNG] Đã đóng gói scaler.joblib tại d:\Hoctap\BTL - Machine Learning\BTL---Machine-Learning---co3117-\models
